# GKE NPI Benchmark Playbook

This runnable playbook guides you through building benchmark images, setting up IAM Workload Identity permissions, connecting to your GKE cluster, and orchestrating NPI benchmarks.

### Instructions
1. Run each cell sequentially.
2. If a cell fails, stop and resolve the error before continuing.

In [ ]:
# --- CONFIGURATION VARIABLES ---
# Replace these with your actual environment details before running any other cells.

PROJECT_ID = "YOUR_PROJECT_ID"
GCSFUSE_VERSION = "v3.5.6"  # Example: v3.5.6
BUCKET_NAME = "YOUR_BUCKET_NAME"
BQ_DATASET_ID = "YOUR_BQ_DATASET_ID"
CLUSTER_NAME = "YOUR_CLUSTER_NAME"
CLUSTER_LOCATION = "YOUR_CLUSTER_LOCATION" # e.g., us-central1-c

## Step 1: Environment Setup & Connect to Cluster
Verify configurations and connect to the GKE cluster.

In [ ]:
!gcloud config set project {PROJECT_ID}
!gcloud container clusters get-credentials {CLUSTER_NAME} --location={CLUSTER_LOCATION} --project={PROJECT_ID}
!kubectl get nodes

## Step 2: Build Benchmark Images
This will use Google Cloud Build to construct the Docker images required for testing and upload them to Artifact Registry.

In [ ]:
!gcloud services enable artifactregistry.googleapis.com --project={PROJECT_ID}
!gcloud artifacts repositories create gcsfuse-benchmarks --repository-format=docker --location=us --project={PROJECT_ID} || echo "Repository might already exist."

!make build PROJECT={PROJECT_ID} GCSFUSE_VERSION={GCSFUSE_VERSION}

!gcloud artifacts docker images list us-docker.pkg.dev/{PROJECT_ID}/gcsfuse-benchmarks

## Step 3: Configure Workload Identity
Create the GSA, bind permissions, and link it to a Kubernetes Service Account (KSA).

In [ ]:
!gcloud iam service-accounts create benchmark-gsa --project={PROJECT_ID} || echo "GSA may already exist."

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:benchmark-gsa@{PROJECT_ID}.iam.gserviceaccount.com" \
    --role="roles/bigquery.dataEditor"

!gcloud storage buckets add-iam-policy-binding gs://{BUCKET_NAME} \
    --member="serviceAccount:benchmark-gsa@{PROJECT_ID}.iam.gserviceaccount.com" \
    --role="roles/storage.objectUser"

!kubectl create serviceaccount benchmark-ksa --namespace default || echo "KSA may already exist."
!gcloud iam service-accounts add-iam-policy-binding benchmark-gsa@{PROJECT_ID}.iam.gserviceaccount.com \
    --role roles/iam.workloadIdentityUser \
    --member "serviceAccount:{PROJECT_ID}.svc.id.goog[default/benchmark-ksa]"

!kubectl annotate serviceaccount benchmark-ksa \
    --namespace default \
    iam.gke.io/gcp-service-account=benchmark-gsa@{PROJECT_ID}.iam.gserviceaccount.com --overwrite

## Step 4: Run Benchmarks Sequentially
We will automatically substitute your variables into the Job YAMLs and run the helper script `run_gke_benchmarks.sh` sequentially.

In [ ]:
import os

os.makedirs('tmp_gke_specs', exist_ok=True)

for filename in os.listdir('gke_pod_specs'):
    if filename.endswith('.yaml'):
        with open(os.path.join('gke_pod_specs', filename), 'r') as file:
            content = file.read()
            
        content = content.replace('YOUR_PROJECT_ID', PROJECT_ID)
        content = content.replace('YOUR_GCSFUSE_VERSION', GCSFUSE_VERSION)
        content = content.replace('YOUR_BQ_DATASET_ID', BQ_DATASET_ID)
        content = content.replace('YOUR_BUCKET_NAME', BUCKET_NAME)
        
        with open(os.path.join('tmp_gke_specs', filename), 'w') as file:
            file.write(content)

with open('run_gke_benchmarks.sh', 'r') as file:
    script_content = file.read()

script_content = script_content.replace('gke_pod_specs', 'tmp_gke_specs')

with open('tmp_run_benchmarks.sh', 'w') as file:
    file.write(script_content)

print("Substituted variables into YAML specs successfully. Running benchmarks...")
!chmod +x tmp_run_benchmarks.sh
!./tmp_run_benchmarks.sh

## Step 5: Clean Up
Remove the temporary generated files. Check your BigQuery dataset for the output!

In [ ]:
import shutil
import os

if os.path.exists('tmp_gke_specs'):
    shutil.rmtree('tmp_gke_specs')
if os.path.exists('tmp_run_benchmarks.sh'):
    os.remove('tmp_run_benchmarks.sh')
print("Cleanup complete.")